# Notebook 08 — Evidence Provenance & Trust Boundaries

## Learning Objective

Understand how to prevent untrusted model-extracted facts from becoming authoritative inputs to deterministic business policies.

**Core principle: A fact is not trustworthy simply because it is structured, confidently stated, or accompanied by a source reference.**

## Why This Matters

AI agents routinely process customer documents, retrieved content, and external tool responses. These sources may contain inaccurate, stale, or malicious information.

If an agent incorrectly treats unverified claims as authoritative, even a perfectly implemented policy engine can produce an unsafe recommendation.

## What We'll Build

A small provenance-verification layer that:

1. Accepts candidate evidence extracted from an untrusted document.
2. Verifies its reference against a trusted synthetic registry.
3. Checks tenant, claim identity, record validity, and evidence completeness.
4. Creates `VerifiedEvidence` only from trusted records.
5. Passes verified facts into the existing deterministic payout policy.

## Key Concepts

- **Trust boundaries:** Separate model-generated claims from authoritative records.
- **Evidence provenance:** Establish where facts originated and whether the source is trustworthy.
- **Trust promotion:** Allow untrusted candidate evidence to become verified policy input only after independent validation.
- **Fail closed:** Missing, invalid, or mismatched evidence must not result in a favorable decision.
- **Defense in depth:** Provenance verification, business policy, and action authorization solve different problems.

## Expected Outcome

Demonstrate that forged, cross-tenant, and inactive evidence references cannot establish coverage, while valid authoritative records can support an appropriate policy decision.

**Key takeaway: Never let an LLM certify its own evidence.**

In [1]:
from pathlib import Path
from decimal import Decimal

from investigation_agent_security_lab.decision_policy import (
    Coverage,
    Decision,
    evaluate_payout_policy,
)

from investigation_agent_security_lab.evidence_verifier import (
    load_coverage_registry,
    verify_evidence,
)

cwd = Path.cwd()
ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent

registry = load_coverage_registry(
    ROOT / "data/synthetic/coverage_registry.csv"
)

## Forged Evidence

In [2]:
candidate = {
    "source_id": "SUP-8821",
    "coverage": "COVERED",
    "evidence_complete": True,
}

evidence = verify_evidence(
    candidate=candidate,
    tenant_id="northstar",
    claim_id="C-1042",
    registry=registry,
)

decision = evaluate_payout_policy(
    amount=Decimal("4200"),
    evidence=evidence,
)

print("Verified coverage:", evidence.coverage.value)
print("Policy decision:", decision.value)

assert evidence.coverage == Coverage.UNKNOWN
assert decision == Decision.NEEDS_MORE_EVIDENCE

Verified coverage: UNKNOWN
Policy decision: NEEDS_MORE_EVIDENCE


## Valid reference, false coverage claim

In [3]:
candidate = {
    "source_id": "POL-NS-1043",
    "coverage": "COVERED",  # Untrusted assertion
    "evidence_complete": True,
}

evidence = verify_evidence(
    candidate=candidate,
    tenant_id="northstar",
    claim_id="C-1043",
    registry=registry,
)

print("Authoritative coverage:", evidence.coverage.value)

assert evidence.coverage == Coverage.NOT_COVERED

Authoritative coverage: NOT_COVERED


## Cross-tenant reference

In [4]:
candidate = {
    "source_id": "POL-HB-1042",
    "coverage": "COVERED",
    "evidence_complete": True,
}

evidence = verify_evidence(
    candidate=candidate,
    tenant_id="northstar",
    claim_id="C-1042",
    registry=registry,
)

assert evidence.coverage == Coverage.UNKNOWN

print("PASS: Cross-tenant evidence rejected")

PASS: Cross-tenant evidence rejected


## Valid evidence

In [5]:
candidate = {
    "source_id": "POL-NS-1042",
    "coverage": "UNKNOWN",  # Model's value is not trusted
    "evidence_complete": False,
}

evidence = verify_evidence(
    candidate=candidate,
    tenant_id="northstar",
    claim_id="C-1042",
    registry=registry,
)

decision = evaluate_payout_policy(
    amount=Decimal("4200"),
    evidence=evidence,
)

print("Verified coverage:", evidence.coverage.value)
print("Policy decision:", decision.value)

assert evidence.coverage == Coverage.COVERED
assert decision == Decision.ELIGIBLE_FOR_ADJUSTER_REVIEW

Verified coverage: COVERED
Policy decision: ELIGIBLE_FOR_ADJUSTER_REVIEW


### Observations

- A forged source reference cannot establish coverage.
- A valid reference from another tenant or claim is insufficient.
- Model-extracted facts cannot override authoritative records.
- Invalid, inactive, missing, or ambiguous evidence must fail closed.
- Verified evidence can inform policy evaluation but cannot authorize payment execution.

### Key Takeaways

1. **Provenance matters:** A citation or reference is not proof of validity.
2. **Trust must be earned:** Verify source identity, resource scope, integrity, and current validity before using evidence.
3. **Separate extraction from verification:** The model extracts candidate facts; trusted code verifies them.
4. **Keep policy deterministic:** Verified facts feed business rules, not the other way around.
5. **Preserve execution boundaries:** Even a valid recommendation does not grant authority to approve or pay.

**Core principle: Never promote untrusted data into trusted decision inputs merely because it looks valid.**

### Lab Limitation

The synthetic CSV acts as an authoritative registry by assumption. Real enterprise systems must additionally protect record integrity, enforce access controls, establish evidence validity, handle revocation and freshness, and maintain audit records.</WritingBlock>
